<a href="https://colab.research.google.com/github/Saibhossain/ArXiv_Research_Assistant/blob/main/PHASE0_Research_Paper_Ingestion%26RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHASE 0 — Research Paper Ingestion + RAG

arXiv paper PDF, the system can read it and answer questions grounded in the paper.


Project Structure:

```
deep-research-agent/
├── phase0/
│   ├── arxiv_search.py
│   ├── pdf_loader.py
│   ├── chunker.py
│   ├── embed_store.py
│   ├── rag_qa.py
│   └── config.py
├── data/
│   ├── papers/
│   └── vector_store/
├── requirements.txt
└── README.md


```



In [1]:
!pip install arxiv pdfplumber sentence-transformers faiss-cpu numpy tqdm torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.1 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=d9493229d9987f2622f884080f79c88b19fecb93bb9ed9893ebef9b6fb5d26c0
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [3]:
# phase0/arxiv_search.py

import arxiv

def search_arxiv(query, max_results=3):
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    papers = []
    for result in search.results():
        papers.append({
            "title": result.title,
            "authors": [a.name for a in result.authors],
            "pdf_url": result.pdf_url,
            "summary": result.summary
        })
    return papers

In [4]:
# test arxiv_search
results = search_arxiv("retrieval augmented generation")
for p in results:
    print(p["title"])

/tmp/ipython-input-860/4058510215.py:13: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
Intelligent Interaction Strategies for Context-Aware Cognitive Augmentation
Factually: Exploring Wearable Fact-Checking for Augmented Truth Discernment


In [5]:
# phase0/pdf_loader.py

import pdfplumber
import requests
from pathlib import Path

def download_pdf(url, save_path):
    r = requests.get(url)
    with open(save_path, "wb") as f:
        f.write(r.content)

def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

In [6]:
Path("data/papers").mkdir(parents=True, exist_ok=True)

In [7]:
# phase0/chunker.py
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [8]:
# phase0/embed_store.py

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

class VectorStore:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.index = None
        self.texts = []

    def add_texts(self, chunks):
        embeddings = self.model.encode(chunks)
        self.texts.extend(chunks)

        if self.index is None:
            dim = embeddings.shape[1]
            self.index = faiss.IndexFlatL2(dim)

        self.index.add(np.array(embeddings))

    def search(self, query, k=5):
        q_emb = self.model.encode([query])
        D, I = self.index.search(np.array(q_emb), k)
        return [self.texts[i] for i in I[0]]

In [10]:
# phase0/rag_qa.py

def answer_question(question, store):
    retrieved_chunks = store.search(question)
    context = "\n".join(retrieved_chunks)

    print("\n--- Retrieved Context ---\n")
    print(context[:1500])

In [11]:
store = VectorStore()
store.add_texts(["RAG combines retrieval and generation for grounded LLMs."])

answer_question("What is RAG?", store)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Retrieved Context ---

RAG combines retrieval and generation for grounded LLMs.
RAG combines retrieval and generation for grounded LLMs.
RAG combines retrieval and generation for grounded LLMs.
RAG combines retrieval and generation for grounded LLMs.
RAG combines retrieval and generation for grounded LLMs.




---


## Full Pipeline Test

```
arXiv → PDF → Text → Chunks → Embeddings → Query → Retrieved Context

```


